In [ ]:
# Make this notebook work from fine-tuning/ or fine-tuning/clinical-rtor/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('clinical-rtor', 'pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 01 (Clinical) · Supervised Fine-Tuning — teach the abstraction rules

The hard part of Return-to-OR abstraction is **judgment under conflict**: a staged washout that still happens within 30 days is *not* a return (Rule 1.1 wins), but a base model sees "returned to OR" + complication words and says `true`. SFT bakes the rule-ordering in. *Same prompt — the tuned model stops over-calling RTOR.*

---
## Step 1 — Config, client, prompt & data

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

load_dotenv()

AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_MODEL               = os.environ.get('BASE_MODEL', 'gpt-4o-mini-2024-07-18')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
SUBSCRIPTION_ID          = os.environ.get('AZURE_SUBSCRIPTION_ID')
RESOURCE_GROUP           = os.environ.get('AZURE_RESOURCE_GROUP')
RESOURCE_NAME            = os.environ.get('AZURE_RESOURCE_NAME')
TENANT_ID                = os.environ.get('AZURE_TENANT_ID')

_cred = DefaultAzureCredential(interactive_browser_tenant_id=TENANT_ID) if TENANT_ID else DefaultAzureCredential()
client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
    api_version             = AZURE_OPENAI_API_VERSION,
)
print('client ready ->', AZURE_OPENAI_ENDPOINT)


In [ ]:
import json
from pathlib import Path

CASES = [json.loads(l) for l in Path('data/rtor_cases.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]

RULES_BLOCK = '''
### SPECIFIC ABSTRACTION RULES

Rule 1 - Conflict-resolution order (apply in this EXACT priority; the FIRST match decides):
  1. Planned / staged overrides everything. If the index OR current operative note documents that
     the second procedure was planned, staged, anticipated, or scheduled at the time of the index
     surgery, then is_return_to_or = false (even if it occurs within 30 days).
  2. Unplanned + related + within 30 days = RTOR. If the current surgery is unplanned and treats a
     complication of the index surgery (bleeding, hematoma, surgical-site infection, wound dehiscence,
     anastomotic leak, abscess, graft/flap failure) within 30 days, then is_return_to_or = true.
  3. Unrelated anatomy or new diagnosis = not RTOR (false), regardless of timing.
  4. Outside the 30-day window = not RTOR (false).

Rule 2 - Operating-room requirement. The return must be to an operating room. Bedside, ICU, IR,
  endoscopy-suite, or clinic procedures do NOT count: is_return_to_or = false.

Rule 3 - Evidence requirement. Quote the single most decisive sentence from the source documents
  verbatim, then state which rule it triggers.
'''

SYSTEM_PROMPT = (
    'You are a surgical-quality abstraction assistant for Acme Health. Determine whether the '
    'current operative episode is an unplanned Return to the Operating Room (RTOR) for the index '
    'surgery, applying the rules below.\n'
    + RULES_BLOCK +
    '\n### TASK EXECUTION\n'
    '- Read the provided text thoroughly.\n'
    '- Evaluate the context against the Specific Abstraction Rules, resolving any conflicting data '
    'using the exact order specified in Rule 1.\n'
    '- Output ONLY a valid JSON object with exactly two keys: "is_return_to_or" (boolean) and '
    '"evidence" (string citing the exact text used and how it applies to the rules).\n'
    '- Do not include conversational filler. Do not include markdown formatting like a json fence.'
)

TEMPLATE = '''Patient Timeline:
{patient_timeline_json}

Progress Note Details:
{progress_note_json}

Index Surgery Procedure Description:
{index_surgery_procedure_desc}

Index Surgery Operative Note:
{index_surgery_op_note}

Current Surgery Procedure Description:
{current_surgery_procedure_desc}

Current Surgery Operative Note:
{current_surgery_op_note}

Task: Determine if the current operating note/surgery represents a return to the operating room based strictly on the abstraction rules provided above. Output ONLY the raw JSON object.'''

def build_user_prompt(case):
    return TEMPLATE.format(
        patient_timeline_json        = json.dumps(case.get('patient_timeline', []), indent=2),
        progress_note_json           = json.dumps(case.get('progress_note', {}), indent=2),
        index_surgery_procedure_desc = case.get('index_surgery_procedure_desc', ''),
        index_surgery_op_note        = case.get('index_surgery_op_note', ''),
        current_surgery_procedure_desc = case.get('current_surgery_procedure_desc', ''),
        current_surgery_op_note      = case.get('current_surgery_op_note', ''),
    )

def safe_parse(val):
    '''Safely extract JSON from the LLM response, stripping stray markdown fences.'''
    try:
        clean = str(val).strip()
        if clean.startswith('```'):
            clean = clean.strip('`')
            if clean.startswith('json'):
                clean = clean[4:]
        return json.loads(clean.strip())
    except Exception as e:
        return {'is_return_to_or': None, 'evidence': f'Parse Error: {e} | Raw: {val}'}

print(f'Loaded {len(CASES)} labeled cases. Prompt + parser ready.')
print('--- USER PROMPT for', CASES[0]['case_id'], '(first 500 chars) ---')
print(build_user_prompt(CASES[0])[:500])


---
## Step 2 — BEFORE: the base model on the conflict case

`RTOR-0013` is a **staged** return that occurs on day 3. Gold = `false` (planned). Watch the base model likely over-call it `true`.

In [ ]:
hard = next(c for c in CASES if c['case_id'] == 'RTOR-0013')
r = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
              {'role': 'user',   'content': build_user_prompt(hard)}],
    temperature=0.0, max_tokens=300, response_format={'type': 'json_object'},
)
pred = safe_parse(r.choices[0].message.content)
print('Case    :', hard['case_id'])
print('GOLD    :', hard['is_return_to_or'])
print('BASE    :', pred.get('is_return_to_or'))
print('Evidence:', pred.get('evidence'))


---
## Step 3 — Submit the SFT job

Idempotent: a `.rtor_sft_job_id` marker prevents duplicate jobs; a stale marker (deleted job) is cleared automatically. Requires `data/rtor_training.jsonl` from Lab 00.

In [ ]:
import time
from pathlib import Path
from openai import NotFoundError

TRAIN = Path('data/rtor_training.jsonl')
VAL   = Path('data/rtor_validation.jsonl')
assert TRAIN.exists(), 'Run Lab 00 first to generate data/rtor_training.jsonl'

_marker = Path('.rtor_sft_job_id')
job_id = None
if _marker.exists():
    cand = _marker.read_text().strip()
    try:
        s = client.fine_tuning.jobs.retrieve(cand)
        job_id = cand
        print('existing job:', job_id, '|', s.status)
    except NotFoundError:
        print('stale marker cleared'); _marker.unlink(missing_ok=True)

if job_id is None:
    up_tr = client.files.create(file=open(TRAIN, 'rb'), purpose='fine-tune')
    up_va = client.files.create(file=open(VAL, 'rb'),  purpose='fine-tune')
    print('uploaded train/val:', up_tr.id, up_va.id)
    for fid in (up_tr.id, up_va.id):
        for _ in range(60):
            if client.files.retrieve(fid).status == 'processed':
                break
            time.sleep(5)
    job = client.fine_tuning.jobs.create(
        training_file   = up_tr.id,
        validation_file = up_va.id,
        model           = BASE_MODEL,
        suffix          = 'acme-rtor',
        seed            = 42,
        hyperparameters = {'n_epochs': 3},
        extra_body      = {'trainingType': 'GlobalStandard'},
    )
    job_id = job.id
    _marker.write_text(job_id)
    print('submitted:', job_id, '|', job.status)


---
## Step 4 — Monitor (self-healing on a 404)

In [ ]:
import time
from pathlib import Path
from openai import NotFoundError

job_id = globals().get('job_id') or (
    Path('.rtor_sft_job_id').read_text().strip() if Path('.rtor_sft_job_id').exists() else None)
print('job_id:', job_id)
try:
    st = client.fine_tuning.jobs.retrieve(job_id) if job_id else None
except NotFoundError:
    st = None

if st is None:
    Path('.rtor_sft_job_id').unlink(missing_ok=True)
    print('job missing (404) — re-run Step 3 to submit a fresh job.')
else:
    while st.status not in ('succeeded', 'failed', 'cancelled'):
        print('  ', st.status, flush=True); time.sleep(30)
        st = client.fine_tuning.jobs.retrieve(job_id)
    print('final:', st.status)
    if st.status == 'succeeded':
        fine_tuned_model = st.fine_tuned_model
        print('fine_tuned_model:', fine_tuned_model)


---
## Step 5 — Deploy the tuned model

**Note:** a deployment bills ~\$1.70/hour even idle — Step 7 tears it down.

In [ ]:
import json, requests, time

FT_DEPLOYMENT_NAME = os.environ.get('FT_DEPLOYMENT_NAME') or 'acme-rtor-deployment'
fine_tuned_model = globals().get('fine_tuned_model')
assert fine_tuned_model, 'No fine_tuned_model yet — finish Step 4 (job must succeed).'

auth = _cred.get_token('https://management.azure.com/.default').token
deploy_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}'
    f'/providers/Microsoft.CognitiveServices/accounts/{RESOURCE_NAME}'
    f'/deployments/{FT_DEPLOYMENT_NAME}'
)
r = requests.put(
    deploy_url, params={'api-version': '2024-10-01'},
    headers={'Authorization': f'Bearer {auth}', 'Content-Type': 'application/json'},
    json={'sku': {'name': 'GlobalStandard', 'capacity': 1},
          'properties': {'model': {'format': 'OpenAI', 'name': fine_tuned_model, 'version': '1'}}},
)
print('PUT', r.status_code, r.reason)
status_url = deploy_url + '?api-version=2024-10-01'
while True:
    s = requests.get(status_url, headers={'Authorization': f'Bearer {auth}'}).json()
    state = s.get('properties', {}).get('provisioningState', 'Unknown')
    print(state)
    if state == 'Succeeded':
        break
    if state in ('Failed', 'Canceled'):
        print(json.dumps(s, indent=2)); break
    time.sleep(20)


---
## Step 6 — AFTER: tuned vs base on the conflict cases

Run a mix of planned, unplanned, bedside, and unrelated cases through both and count agreement with the gold label.

In [ ]:
ids = ('RTOR-0013', 'RTOR-0002', 'RTOR-0009', 'RTOR-0001', 'RTOR-0006', 'RTOR-0010')
EVAL_CASES = [c for c in CASES if c['case_id'] in ids]

def predict(dep, case):
    r = client.chat.completions.create(
        model=dep,
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user',   'content': build_user_prompt(case)}],
        temperature=0.0, max_tokens=300, response_format={'type': 'json_object'},
    )
    return safe_parse(r.choices[0].message.content).get('is_return_to_or')

print(f"{'case':12} {'gold':6} {'base':6} {'tuned':6}")
bc = tc = 0
for c in EVAL_CASES:
    b = predict(BASE_DEPLOYMENT, c)
    t = predict(FT_DEPLOYMENT_NAME, c)
    bc += int(b == c['is_return_to_or'])
    tc += int(t == c['is_return_to_or'])
    print(f"{c['case_id']:12} {str(c['is_return_to_or']):6} {str(b):6} {str(t):6}")
print(f"\nBase  accuracy: {bc}/{len(EVAL_CASES)}")
print(f"Tuned accuracy: {tc}/{len(EVAL_CASES)}")


---
## Step 7 — Cleanup (stop the hourly deployment bill, keep the model)

In [ ]:
import requests
auth = _cred.get_token('https://management.azure.com/.default').token
url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}'
    f'/providers/Microsoft.CognitiveServices/accounts/{RESOURCE_NAME}'
    f'/deployments/{globals().get("FT_DEPLOYMENT_NAME", "acme-rtor-deployment")}'
)
r = requests.delete(url, params={'api-version': '2024-10-01'},
                    headers={'Authorization': f'Bearer {auth}'})
print('deployment delete:', r.status_code)
print('fine-tuned model kept:', globals().get('fine_tuned_model'))


---
## Takeaways

- SFT injects **judgment**, not just facts — the tuned model honors Rule 1's planned-overrides-everything ordering the base model trips on.
- Same `messages` mechanics as the member-services SFT lab; only the data changed.
- Next: **Lab 03** runs the abstractor over a whole batch and shows the tool-schema token cost you can fine-tune away.